# REVISED MASTER · One-seed Cat/Dog causal pilot

Upload this notebook to Google Colab, select a GPU, and choose **Runtime → Run all**. Authorize Drive and upload the **new matching project ZIP** when requested. Upload the `.ipynb` file directly—do not copy formatted chat text into a code cell.

**Fixed scope:** original CIFAR-10 Cat/Dog splits and MLP; seed 42; final-hidden neuron masks and output-weight masks; 10%/40% budgets; targeted plus three random controls; linear probes; same-input neuron-activation restoration. Maximum baseline epochs 30 (patience 5); maximum probe epochs 20 (patience 4).

This is **32 ablation conditions and 33 linear probe fits**, with eight main **600-DPI PNG** figures. Output-weight probes are unaffected-hidden-state references, explicitly distinguished from probes of modified hidden states. Whole-layer patching is a sanity check for restored computation, not discovery of a unique circuit.

The experiment cell has a 45-minute execution budget by default so that a one-hour session has some room for setup/reporting. **Completion within one hour is not guaranteed**; setup/download time and assigned hardware vary. Completed work is saved to Drive and reused next session. There are no synthetic research results or cross-seed inference claims.

## 1 · Environment and free GPU/CPU detection
Keep Colab’s preinstalled matched PyTorch/torchvision packages. Install the remaining project dependencies from PyPI.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
os.environ.setdefault('MPLCONFIGDIR', '/content/loi_matplotlib_cache')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
               'numpy>=1.23', 'scipy>=1.10', 'scikit-learn>=1.2', 'pandas>=1.5',
               'matplotlib>=3.8', 'PyYAML>=6', 'pytest>=7', 'Pillow>=9'], check=True)
if importlib.util.find_spec('torch') is None or importlib.util.find_spec('torchvision') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision'], check=True)
try:
    import torch, torchvision
except Exception as exc:
    raise RuntimeError('PyTorch/torchvision did not import. Use a fresh standard Colab runtime with its matched packages, then rerun.') from exc
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU fallback selected. The complete study will be much slower.')


## 2 · Mount Drive and set persistent storage
Data, uploaded project archives, configuration, checkpoints and outputs are kept in your own Drive. Authorize the mount yourself. On a later runtime, use the same account and storage path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/lost-or-inaccessible')
DATA_DIR = DRIVE_PROJECT / 'datasets'
OUTPUT_DIR = DRIVE_PROJECT / 'revised_results'
UPLOAD_DIR = DRIVE_PROJECT / 'uploads'
EXPORT_DIR = DRIVE_PROJECT / 'exports'
for path in [DATA_DIR, OUTPUT_DIR, UPLOAD_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('Persistent project folder:', DRIVE_PROJECT)


## 3 · Upload/reuse the project ZIP

On the first run, select the supplied **updated** `lost-or-inaccessible-revised.zip`. The archive is copied to Drive and its SHA-256 recorded. On resumed sessions the same ZIP is reused automatically. Set `UPLOAD_NEW_ZIP = True` only when you deliberately want a different source version; source changes create a different experiment directory.

The source is extracted into the temporary Colab runtime. Existing dataset/checkpoint folders in an uploaded ZIP are ignored; the verified Drive dataset and run outputs are managed separately.

In [ ]:
import hashlib, json, shutil, stat, zipfile
from pathlib import PurePosixPath
from google.colab import files
UPLOAD_NEW_ZIP = False
ZIP_PATH_OVERRIDE = ''  # Optional existing .zip path in your mounted Drive.
last_zip = UPLOAD_DIR / 'LAST_REVISED_PROJECT_ZIP.txt'
if ZIP_PATH_OVERRIDE:
    project_zip = Path(ZIP_PATH_OVERRIDE)
elif last_zip.exists() and not UPLOAD_NEW_ZIP:
    project_zip = Path(last_zip.read_text().strip())
else:
    print('Select lost-or-inaccessible-revised.zip supplied with this revised notebook.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one project ZIP. Rerun this cell to try again.')
    name, payload = next(iter(uploaded.items()))
    if not name.lower().endswith('.zip'):
        raise ValueError('The project must be a .zip file.')
    digest = hashlib.sha256(payload).hexdigest()
    project_zip = UPLOAD_DIR / f'project_{digest[:16]}.zip'
    if not project_zip.exists():
        temporary = project_zip.with_suffix('.zip.partial')
        temporary.write_bytes(payload)
        temporary.replace(project_zip)
    last_zip.write_text(str(project_zip))
    del payload, uploaded
if not project_zip.is_file():
    raise FileNotFoundError(f'ZIP not found: {project_zip}. Set UPLOAD_NEW_ZIP=True to upload again.')
with project_zip.open('rb') as stream:
    hasher = hashlib.sha256()
    for block in iter(lambda: stream.read(1024*1024), b''):
        hasher.update(block)
    digest = hasher.hexdigest()
WORK_ROOT = Path('/content/loi_sources') / digest[:16]
WORK_ROOT.mkdir(parents=True, exist_ok=True)
extraction_marker = WORK_ROOT / '.extraction_complete'
if not extraction_marker.exists():
    skip = {'__MACOSX', '__pycache__', '.git', '.venv', '.pytest_cache', 'datasets', 'results'}
    with zipfile.ZipFile(project_zip) as archive:
        members = []
        for item in archive.infolist():
            rel = PurePosixPath(item.filename)
            if rel.is_absolute() or '..' in rel.parts or '\\\\' in item.filename:
                raise ValueError(f'Unsafe ZIP member: {item.filename}')
            if stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError('Symlinks are not accepted in the project ZIP.')
            if skip.intersection(rel.parts) or item.is_dir():
                continue
            members.append(item)
        size = sum(item.file_size for item in members)
        if size > shutil.disk_usage(WORK_ROOT).free:
            raise RuntimeError('Insufficient Colab runtime disk space to extract the source ZIP.')
        for item in members:
            destination = WORK_ROOT.joinpath(*PurePosixPath(item.filename).parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, destination.open('wb') as target:
                shutil.copyfileobj(source, target)
    extraction_marker.write_text(digest)
candidates = [p.parent for p in WORK_ROOT.rglob('run.py')
              if (p.parent/'configs/colab.yaml').exists() and (p.parent/'models/mlp.py').exists()]
if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one research project in the ZIP; found {len(candidates)}.')
PROJECT_DIR = candidates[0]
if not (PROJECT_DIR/'configs/revised.yaml').exists():
    raise RuntimeError('Upload the updated revised project ZIP; this archive has no revised profile.')
if '--report-only' not in (PROJECT_DIR/'run.py').read_text() or 'PNG-only' not in (PROJECT_DIR/'analysis/plots.py').read_text():
    raise RuntimeError('This ZIP is an older version. Set UPLOAD_NEW_ZIP=True and upload the updated supplied project ZIP.')
print('Source ZIP:', project_zip)
print('SHA-256:', digest)
print('Project directory:', PROJECT_DIR)


## 4 · Preflight software tests
These tests use synthetic fixtures strictly to verify the software. They are not research observations. Failure stops the notebook before the main study. Fixture plots use low DPI to keep this check quick; research figures are rendered at 600 DPI below.

In [ ]:
test_environment = dict(os.environ, LOI_FIGURE_DPI='100')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_DIR,
               env=test_environment, check=True)
print('Software preflight passed. Research experiments have not started yet.')


## 5 · Download/reuse and verify official CIFAR-10
If you already ran the Drive-download notebook, this reuses `MyDrive/lost-or-inaccessible/datasets`. The complete original files remain unchanged. Only Cats (3→0) and Dogs (5→1) enter the research pipeline. A verification record is written only after the official file checks pass.

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from torchvision.datasets import CIFAR10
train = CIFAR10(str(DATA_DIR), train=True, download=True)
test = CIFAR10(str(DATA_DIR), train=False, download=True)
assert train._check_integrity() and test._check_integrity()
train_counts, test_counts = Counter(train.targets), Counter(test.targets)
assert train_counts[3] == train_counts[5] == 5000
assert test_counts[3] == test_counts[5] == 1000
verification = dict(status='verified', verified_at_utc=datetime.now(timezone.utc).isoformat(),
                    source=CIFAR10.url, cat_dog_train=10000, cat_dog_test=2000,
                    torchvision_file_checksums='passed', directory=str(DATA_DIR))
(DATA_DIR/'DATASET_VERIFIED.json').write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))
del train, test


## 6 · Confirm the revised pilot settings
The printed YAML is the actual executed configuration, not the old full study. Both target classes are retained. Rankings use only analysis data; fitting/selecting models and probes never uses official test examples. No nonlinear probes, other hidden intervention layers, global-edge sweep or complete-circuit search is run.

In [ ]:
import yaml
base = yaml.safe_load((PROJECT_DIR/'configs/base.yaml').read_text())
base.update(yaml.safe_load((PROJECT_DIR/'configs/colab.yaml').read_text()))
base.update(yaml.safe_load((PROJECT_DIR/'configs/revised.yaml').read_text()))
assert base['seeds'] == [42]
assert base['neuron_layers'] == [2] and base['edge_layers'] == [3]
assert base['probe_families'] == ['linear'] and base['report_mode'] == 'revised'
print(yaml.safe_dump(base, sort_keys=False))
print('32 ablation conditions: 16 neuron masks + 16 output-weight masks.')
print('33 linear probe fits: 1 clean + 16 modified-hidden + 16 unchanged-hidden references.')
print('16 same-input activation patches; 8 measured PNG figure families.')
print('One seed cannot provide cross-seed standard deviations or model-population confidence intervals.')


## 7 · Run the revised experiment with Drive-backed resume
This executes the reduced profile in dependency order, including baseline, ranking, 32 ablations, comparison, 33 linear probes, same-input activation restoration, and within-model statistics. The wall-clock guard applies to this experiment cell; setup/download/reporting also take time. A guard stop saves completed work and proceeds to report available outputs; incomplete stages remain labeled incomplete. Baseline training checkpoints are saved at each epoch, so an interrupted epoch is rerun on resume.

In [ ]:
import queue, threading, time
EXPERIMENT = 'all'
FIGURE_DPI = 600
MAX_EXPERIMENT_MINUTES = 45  # Explicit wall-clock guard; rerun to resume if needed.
command = [sys.executable, '-u', 'run.py', '--profile', 'colab',
           '--config', 'configs/revised.yaml', '--experiment', EXPERIMENT,
           '--device', DEVICE, '--data-root', str(DATA_DIR), '--output', str(OUTPUT_DIR),
           '--download', '--skip-report', '--figure-dpi', str(FIGURE_DPI)]
log_file = DRIVE_PROJECT/'revised_pipeline_latest.log'
print('Command:', ' '.join(command))
print('Persistent log:', log_file)
RUN = None
started = time.monotonic()
process = subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
lines = queue.Queue()
def read_console():
    for line in process.stdout:
        lines.put(line)
    lines.put(None)
threading.Thread(target=read_console, daemon=True).start()
time_limit_reached = False
try:
    with log_file.open('a', buffering=1) as log:
        log.write('\n=== REVISED SESSION START ' + datetime.now(timezone.utc).isoformat() + ' ===\n')
        while True:
            if time.monotonic()-started > MAX_EXPERIMENT_MINUTES*60:
                time_limit_reached = True
                process.terminate()
                try: process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
                print('Time budget reached. Completed checkpoints/conditions remain saved; resume next session.')
                break
            try: line = lines.get(timeout=1)
            except queue.Empty: continue
            if line is None: break
            print(line, end='', flush=True); log.write(line)
            if line.startswith('Run directory: '):
                RUN = Path(line.split('Run directory: ',1)[1].split(' | device:',1)[0].strip())
                (DRIVE_PROJECT/'LAST_REVISED_RUN.txt').write_text(str(RUN))
        return_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    try: process.wait(timeout=15)
    except subprocess.TimeoutExpired: process.kill(); process.wait()
    raise RuntimeError('Interrupted. Completed artifacts remain on Drive; rerun the identical revised notebook to resume.')
if return_code and not time_limit_reached:
    raise RuntimeError(f'Experiment failed with exit code {return_code}; inspect {log_file} and *.failed.json.')
if RUN is None:
    raise RuntimeError('No run directory was recorded. Inspect the log before continuing.')
print('Elapsed experiment minutes:', round((time.monotonic()-started)/60,2))
print('Next cell checks completion and renders available measured figures, even after the time guard.')


## 8 · Verify completion and render the PNG report — E10
This can also be rerun after an interruption to inspect a partial run. It reads existing measurements and does not train models. Missing experiments remain marked incomplete; missing figure families are never fabricated.

In [ ]:
if 'RUN' not in globals() or RUN is None:
    pointer = DRIVE_PROJECT/'LAST_REVISED_RUN.txt'
    if not pointer.exists():
        raise RuntimeError('No run was recorded. Execute the experiment cell first.')
    RUN = Path(pointer.read_text().strip())
if not (RUN/'config.json').exists():
    raise RuntimeError(f'Not a valid run directory: {RUN}')
run_config = json.loads((RUN/'config.json').read_text())
import pandas as pd
records = []
for seed in run_config['seeds']:
    for stage in [f'e{i}' for i in range(1, 9)]:
        marker = RUN/f'seed_{seed}'/f'{stage}.done.json'
        records.append(dict(seed=seed, stage=stage.upper(), status='COMPLETE' if marker.exists() else 'NOT COMPLETE'))
completion = pd.DataFrame(records)
display(completion.pivot(index='seed', columns='stage', values='status'))
subprocess.run([sys.executable, '-u', 'run.py', '--report-only', str(RUN),
                '--figure-dpi', str(FIGURE_DPI)], cwd=PROJECT_DIR, check=True)
full_complete = bool((completion.status=='COMPLETE').all() and (RUN/'e9.done.json').exists())
print('E9 statistics:', 'COMPLETE' if (RUN/'e9.done.json').exists() else 'NOT COMPLETE')
print('REVISED PILOT:', 'COMPLETE' if full_complete else 'INCOMPLETE — do not present partial results as the revised pilot')
figure_status = json.loads((RUN/'figures/figure_status.json').read_text())
print('Figure families:', json.dumps(figure_status['families'], indent=2))
print('PNG directory:', RUN/'figures')


## 9 · Inspect the eight PNG figures and measured tables
P1 training curves; P2 baseline confusion/confidence; P3 final-hidden selectivity; P4 neuron ablation; P5 output-edge ablation; P6 probes and original readout; P7 functional loss versus modified-hidden-state recoverability; P8 activation restoration. Every completed family is exported at 600 DPI. Missing source experiments produce missing figures, not placeholders. The gallery embeds small previews; downloaded originals stay full-resolution. Check labels and the layout audit before sharing figures.

In [ ]:
from IPython.display import display, Image, Markdown
from PIL import Image as PILImage
import io
png_files = sorted((RUN/'figures').glob('*.png'))
if not png_files:
    raise RuntimeError('No PNG figures found; inspect the reporting cell output.')
manifest = json.loads((RUN/'figures/figure_manifest.json').read_text())
figure_rows = []
for path in png_files:
    with PILImage.open(path) as im:
        width, height = im.size
    record = manifest.get(path.name, {})
    figure_rows.append(dict(file=path.name, width_px=width, height_px=height,
                            dpi=record.get('dpi'), layout_flags=len(record.get('text_outside_layout', []))))
display(pd.DataFrame(figure_rows))
for path in png_files:
    display(Markdown('### ' + path.stem.replace('_', ' ')))
    with PILImage.open(path) as original:
        preview = original.copy()
        preview.thumbnail((1600, 1100))
        buffer = io.BytesIO()
        preview.save(buffer, format='PNG')
    display(Image(data=buffer.getvalue(), width=min(1100, preview.width)))
for table in ['T2', 'T4', 'T6', 'T8', 'T9', 'T10']:
    path = RUN/'tables'/f'{table}.csv'
    if path.exists() and path.stat().st_size > 2:
        display(Markdown('### ' + table + ' — first 15 measured rows'))
        display(pd.read_csv(path).head(15))


## 10 · Create download bundles
`FIGURES_PNG_600DPI.zip` contains **only PNG image files**. The separate report bundle contains the figure manifest, measured CSV tables, run configuration, completion records and relevant statistics. Original checkpoints/probes remain on Drive and are not forced into a huge browser download.

The bundles are always saved to Drive. Browser downloads are requested by default; set `DOWNLOAD_TO_COMPUTER = False` to retain only the Drive copies.

In [ ]:
DOWNLOAD_TO_COMPUTER = True
figures_zip = EXPORT_DIR / f'{RUN.name}_FIGURES_PNG_{FIGURE_DPI}DPI.zip'
report_zip = EXPORT_DIR / f'{RUN.name}_MEASURED_REPORT.zip'
with zipfile.ZipFile(figures_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in png_files:
        archive.write(path, path.name)
with zipfile.ZipFile(report_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((RUN/'tables').glob('*')):
        if path.is_file():
            archive.write(path, str(path.relative_to(RUN)))
    for path in sorted((RUN/'figures').glob('*.json')):
        archive.write(path, str(path.relative_to(RUN)))
    for name in ['config.json','environment.json','splits.json','execution_status.json',
                 'e9.done.json','e10.done.json','robustness.csv','paired_tests.csv',
                 'paired_image_bootstrap.csv','circuit_stability.csv','performance_matched_probes.csv']:
        path=RUN/name
        if path.exists():
            archive.write(path, name)
    for seed in run_config['seeds']:
        for path in sorted((RUN/f'seed_{seed}').glob('e*.csv')):
            archive.write(path, str(path.relative_to(RUN)))
export_status = dict(run=str(RUN), revised_pilot_complete=full_complete, figure_format='PNG',
                     figure_dpi=FIGURE_DPI, number_of_png_figures=len(png_files),
                     figures_zip=str(figures_zip), measured_report_zip=str(report_zip))
(EXPORT_DIR/f'{RUN.name}_EXPORT_STATUS.json').write_text(json.dumps(export_status, indent=2))
print(json.dumps(export_status, indent=2))
if DOWNLOAD_TO_COMPUTER:
    files.download(str(figures_zip))
    files.download(str(report_zip))
print('Finished. Inspect completion status before writing any scientific conclusions.')


## Resume and interpretation
Reconnect to Colab, use the same Drive account and revised notebook, keep `UPLOAD_NEW_ZIP=False`, then Run all again. Source and configuration fingerprints keep different runs separate. `LAST_REVISED_RUN.txt` points to this pilot, independently of the original full-study notebook.

A one-hour finish is not guaranteed; the default 45-minute experiment guard leaves some time for reporting. If stopped, the completion grid and figure status remain honest. Software tests use synthetic fixtures only; research results always use CIFAR-10.

Output-edge ablation leaves the final hidden state unchanged. Its probe is an accessibility reference. Neuron ablation changes the state actually probed. Same-input full-layer patching restores the original readout computation and is expected to recover baseline; this is not evidence of a unique minimal circuit. One training seed supports no across-model confidence interval. The notebook has not yet run inside your authenticated Colab account.